In [ ]:
import glob # For file pattern matching
import os # For handling file paths
# Define the directory containing the probe data
PROBES_DIR = os.path.join("C:\\Users\\陳懷浯\\OneDrive\\桌面\\biogas plant\\test")
print(f"PROBES_DIR: {PROBES_DIR}")
print(f"os.path.basename(PROBES_DIR): {os.path.basename(PROBES_DIR)}")


In [ ]:
glob.glob(os.path.join(PROBES_DIR, "Sensor 4*.csv"))[:5] 
# List the first 5 CSV files in the PROBES_DIR that start with "Sensor 4"

In [ ]:
import pandas as pd
import numpy as np

csv_path = os.path.join(PROBES_DIR, "Sensor 4 Sample P.1 Baseline.csv") # Path to the specific CSV file to read
print(f"csv_path basename: {os.path.basename(csv_path)}") # Print the base name of the CSV file
print(f"csv_path dirname: {os.path.dirname(csv_path)}") # Print the directory name of the CSV file
print(os.path.splitext(os.path.basename(csv_path)))# Print the base name and extension of the CSV file
df = pd.read_csv(csv_path)# Read the CSV file into a DataFrame

df.head()# Display the first few rows of the DataFrame

In [ ]:
dfd = df.drop(columns=df.columns[0], inplace=False)
# Drop the first column of the DataFrame, which is often an index or timestamp
dfd.head()# Display the first few rows of the modified DataFrame without the first column

In [ ]:
df.index.values# Display the index values of the DataFrame, which may represent timestamps or row numbers

In [ ]:
wl_range = df.columns[1:].values# Get the column names starting from the second column, which may represent wavelength ranges
wl_range = np.array(wl_range, dtype=int)# Convert the column names to integers, assuming they represent wavelength ranges
wl_range

### Data Loading

#### Read and process csv

In [ ]:
def read_csv_data(path):
    df = pd.read_csv(path, encoding='ISO-8859-1')# Read the CSV file into a DataFrame with the specified encoding
    # Modification to the df
    df.drop(columns=df.columns[0], inplace=True)
    # average the intensity of the same wavelength
    wavelength = df.columns.to_numpy(dtype=int)
    intensity = df.mean(axis=0).to_numpy(dtype=float)
    return wavelength, intensity

# TODO: deal with empty cuvette, lamp on/off, etc.
def get_csv_paths(probes_dir, sensor, reference_sample: bool = False):
    if reference_sample:
        patterns = [
            os.path.join(probes_dir, f"{sensor}_lamp_on.csv"),
            os.path.join(probes_dir, f"{sensor}_lamp_off.csv"),
        ]
        paths = []
        for pattern in patterns:
            paths.extend(glob.glob(pattern))
        return paths
    else:
        pattern = os.path.join(probes_dir, f"{sensor} Sample*.csv")
        return glob.glob(pattern)

In [ ]:
wavelength, intensity = read_csv_data(os.path.join(PROBES_DIR, "Sensor 4 Sample P.1 Baseline.csv"))
print(f"Wavelength length of sensor 4: {len(wavelength)}\n")# Print the length of the wavelength array for sensor 4
print(f"Intensity length: {len(intensity)}")

In [ ]:
print(f"First 5 wavelength/intensity pairs:")
for i in range(min(5, len(wavelength))):# Loop through the first 5 indices (or fewer if there are less than 5) and print the wavelength and intensity pairs, formatted to 2 decimal places
    print(f"  WL: {wavelength[i]}, Intensity: {intensity[i]:.2f}")# Print the first 5 wavelength and intensity pairs, formatted to 2 decimal places

print(f"Last 5 wavelength/intensity pairs:")
for i in range(max(0, len(wavelength) - 5), len(wavelength)):# Loop through the last 5 indices (or fewer if there are less than 5) and print the wavelength and intensity pairs, formatted to 2 decimal places
    print(f"  WL: {wavelength[i]}, Intensity: {intensity[i]:.2f}")# Print the last 5 wavelength and intensity pairs, formatted to 2 decimal places

print(f"Wavelength range: {wavelength[0]} - {wavelength[-1]}")
print(f"Intensity range: {intensity[0]:.2f} - {intensity[-1]:.2f}")


#### Load and Combine all the data

In [ ]:
# Debug: Test glob pattern
test_sensor = "Sensor 2"
test_pattern = os.path.join(PROBES_DIR, f"{test_sensor} Sample*.csv")
print(f"Test pattern: {test_pattern}")
test_files = glob.glob(test_pattern)
print(f"Files found: {test_files}")
print(f"Number of files: {len(test_files)}")

In [ ]:
PROBE_LIST = ["Probe A", "Probe B", "Probe C", "Probe D"]
# Manual order, because Sensor 2 has the lowest wavelength
SENSOR_LIST = ["Sensor 2", "Sensor 1", "Sensor 4", "Sensor 3"]
get_reference_sample = False

all_sensor_data = {}
for sensor in SENSOR_LIST:
    # Initialize the sensor data list, to store the intensity data of each sample seperately
    sensor_data = []
    sample_paths = get_csv_paths(PROBES_DIR, sensor, get_reference_sample)
    print(f"Debug: Processing {sensor}, found {len(sample_paths)} files")
    for csv_path in sample_paths:
        _, intensity = read_csv_data(csv_path)
        sensor_data.append(intensity)

    # Combine the intensity data of all samples into a single array
    if len(sensor_data) > 0:
        all_sensor_data[sensor] = np.vstack(sensor_data)
        print(f"Sensor Data Shape of {sensor}: {np.shape(all_sensor_data[sensor])} ({len(sample_paths)} samples)") 
        # Print the shape of the combined sensor data array, which should be (number of samples, number of wavelengths)
    else:
        print(f"Warning: No data found for {sensor}")

### Plotting

In [ ]:
import matplotlib.pyplot as plt
# Sample data
x = np.linspace(0, 10, 100)
y1 = 5*np.sin(x)
y2 = np.cos(x)
# Create figure and axes
# fig = plt.figure(figsize=(6, 4), dpi=150)
fig, axes = plt.subplots(2,2, dpi=150)
axes[0,0].plot(x, y1, label='sin(x)')
axes[0,0].plot(x, y2, label='cos(x)')
axes[0,0].legend()
axes[0,0].set_title('sin(x) and cos(x)')
axes[0,0].set_xlabel('x')
axes[0,0].set_ylabel('y')

In [ ]:

from typing import Any
import matplotlib.pyplot as plt
import matplotlib.cm as cm

font_size_title  = 10
font_size_ylabel = 8
font_size_xlabel = 8
font_size_ticks  = 6
font_size_legend = 6
grid_line_width  = 0.5
plot_line_width  = 0.5

fig, axes = plt.subplots(2, 2, figsize=(7.00,3.20), dpi=150)
axes = axes.ravel()

for ax, sensor in zip(axes, SENSOR_LIST):
    sample_paths = get_csv_paths(PROBES_DIR, sensor)
    # Create a color map for different files
    colors = cm.get_cmap('tab10')(range(len(sample_paths)))
    
    for idx, path in enumerate(sample_paths):
        wavelength, intensity = read_csv_data(path)
        # Extract filename for legend
        filename = os.path.basename(path)
        ax.plot(wavelength, intensity, linewidth=plot_line_width, 
                color=colors[idx], label=filename, alpha=0.8)
    
    ax.set_title(sensor)
    ax.set_xlabel("Wavelength (nm)")
    ax.set_ylabel("Intensity")
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.tick_params(axis='both', which='major', labelsize=font_size_ticks)
    ax.grid(True, linewidth=grid_line_width, ls='--', alpha=0.3)
    ax.legend(fontsize=font_size_legend, loc='best')

fig.tight_layout()
plt.show()



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean
import pandas as pd

# Calculate similarity metrics for all sensor data
similarity_results = {}

for sensor in SENSOR_LIST:
    if sensor in all_sensor_data:
        data = all_sensor_data[sensor]
        
        # Cosine similarity between samples
        cosine_sim = cosine_similarity(data)
        
        # Euclidean distance between samples
        euclidean_dist = np.array([[euclidean(data[i], data[j]) 
                                     for j in range(len(data))] 
                                    for i in range(len(data))])
        
        similarity_results[sensor] = {
            'cosine_similarity': cosine_sim,
            'euclidean_distance': euclidean_dist
        }
        
        print(f"\n{sensor} - Cosine Similarity Matrix:")
        print(cosine_sim)
        print(f"\n{sensor} - Euclidean Distance Matrix:")
        print(euclidean_dist)

### Statistics 

#### Maximum and Minimum Value
The intensity value of the sensor output should be further normalized based on the data of lamp on/off
